# Windi — thử giọng sách nói nhỏ (Piper/VITS)
Một người nói · tiếng Việt · ONNX 22.050 Hz · train trên Colab, đọc sách local trên Mac.

**Cấu hình CapCut sạch:** không thêm denoise, EQ, compressor, đổi cao độ hoặc tốc độ. Giữ bản thu gốc; chỉ đổi sample rate/mono cho đúng kiến trúc. Giữ hơi thở và phần nghỉ trong clip, tắt trim_silence khi train. Learning rate thử 3e-5, chưa coi là cấu hình tối ưu trước khi nghe file thật. Kiến trúc và số tham số model giữ nguyên.

Notebook này **không tự train**, không tự đăng model, không dùng API chép lời. Khi có file, chỉ đưa lên bản thu bạn chọn. Bản gốc được giữ riêng. Chép lời bằng faster-whisper chạy trong máy Colab; cần nghe/sửa và duyệt trước khi train.

**Trạng thái ban đầu:** chưa có bản thu, chưa có giọng đã train. Model nền tiếng Việt là VAIS1000 medium, không phải giọng riêng đã đạt chất lượng.

Nguồn: [Piper](https://github.com/OHF-Voice/piper1-gpl), [train/export](https://github.com/OHF-Voice/piper1-gpl/blob/main/docs/TRAINING.md), [checkpoint](https://huggingface.co/datasets/rhasspy/piper-checkpoints/tree/main/vi/vi_VN/vais1000/medium).
Piper runtime hiện tại GPL-3.0; model card VAIS1000 ghi dữ liệu CC BY 4.0. Lưu attribution đi kèm thử nghiệm; chưa tích hợp hay phát hành runtime này trong app.


## 1. Cài môi trường riêng
Có thể chạy bước setup bằng CPU. Để train: Runtime → Change runtime type → T4 GPU nếu được cấp. Không chọn TPU. Mỗi máy Colab mới cần chạy lại setup; không coi máy ảo là nơi lưu trữ lâu dài.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, urllib.request
ROOT = Path('/content/windi-small-voice')
ROOT.mkdir(exist_ok=True)
VENV = ROOT/'venv'
PY = VENV/'bin/python'
def run(command, **kwargs):
    p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,**kwargs)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('Lệnh setup thất bại: '+repr(command))
if not (VENV/'bin/pip').exists():
    run([sys.executable,'-m','pip','install','virtualenv==20.35.3'])
    run([sys.executable,'-m','virtualenv','--system-site-packages',str(VENV)])
run([str(PY),'-m','pip','install','--disable-pip-version-check',
    'piper-tts[train]==1.8.0','faster-whisper==1.2.1','av==16.0.1','soundfile==0.13.1'])
subprocess.run(['ffmpeg','-version'],check=True,stdout=subprocess.DEVNULL)
# Official wheel includes the training Python modules, but not the Cython aligner.
align_setup = r"""
import hashlib, importlib.util, shutil, subprocess, sys, urllib.request
from pathlib import Path
align=Path(importlib.util.find_spec('piper').origin).parent/'train/vits/monotonic_align'
target=align/'monotonic_align'; target.mkdir(exist_ok=True)
if not list(target.glob('core*.so')):
    url='https://raw.githubusercontent.com/OHF-Voice/piper1-gpl/619f513580c04dff5ef9a680f6f3f6979ed94d2e/src/piper/train/vits/monotonic_align/core.pyx'
    data=urllib.request.urlopen(url,timeout=60).read()
    if hashlib.sha256(data).hexdigest()!='8640b303683823a4a1259179547ef476999b1cbb2e46ff656b970763cfbc1157':
        raise RuntimeError('Mã aligner không đúng checksum.')
    (align/'core.pyx').write_bytes(data)
    subprocess.run([sys.executable,'-m','Cython.Build.Cythonize','-i','core.pyx'],cwd=align,check=True)
    for so in align.glob('core*.so'): shutil.copy2(so,target/so.name)
print('Cython aligner đã build.')
exporter=Path(importlib.util.find_spec('piper').origin).parent/'train/export_onnx.py'
source=exporter.read_text()
if 'dynamo=False' not in source:
    if source.count('opset_version=OPSET_VERSION,') != 1:
        raise RuntimeError('Exporter khác cấu trúc đã kiểm tra.')
    exporter.write_text(source.replace('opset_version=OPSET_VERSION,',
        'opset_version=OPSET_VERSION,\n        dynamo=False,'))
print('Giữ legacy ONNX exporter cho kiến trúc Piper VITS trên Torch mới.')
"""
run([str(PY),'-c',align_setup])
print('Đã cài môi trường riêng. Chưa bắt đầu train.')


## 2. Công cụ tách, duyệt và train — mã đi kèm notebook

In [ ]:
TOOLKIT = '"""Isolated training tools. Never imported by the audiobook runtime."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport wave\nfrom array import array\n\nPROFILE = dict(id=\'capcut-clean-narration-v1\', sampleRate=22050,\n               trimSilence=False, learningRate=3e-5, pilotSteps=300,\n               effects=[], preserveOriginal=True, target=\'speaker-and-reading-style\')\n\n\ndef word_groups(segments, max_seconds=18):\n    """Keep all ASR words in order; prefer punctuation and natural pause boundaries."""\n    pending = []\n    for segment in segments:\n        for word in segment.words or []:\n            if pending and word.start - pending[-1].end > .65:\n                yield pending\n                pending = []\n            if pending and word.end - pending[0].start > max_seconds:\n                # Prefer a completed clause to cutting inside the next phrase.\n                cuts = [i+1 for i,w in enumerate(pending)\n                        if w.word.rstrip().endswith((\',\', \';\', \':\', \'.\', \'?\', \'!\', \'…\'))\n                        and w.end-pending[0].start >= 3]\n                cut = cuts[-1] if cuts else len(pending)\n                yield pending[:cut]\n                pending = pending[cut:]\n                if pending and word.end-pending[0].start > max_seconds:\n                    yield pending\n                    pending = []\n            pending.append(word)\n            if word.word.rstrip().endswith((\'.\', \'?\', \'!\', \'…\')) and word.end-pending[0].start >= 2:\n                yield pending\n                pending = []\n    if pending:\n        yield pending\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.partial\')\n    with tmp.open(\'w\', encoding=\'utf-8\') as f:\n        json.dump(value, f, ensure_ascii=False, indent=2)\n        f.flush()\n        os.fsync(f.fileno())\n    tmp.replace(path)\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef safe_path(root, relative):\n    root = Path(root).resolve()\n    path = (root / relative).resolve()\n    if not path.is_relative_to(root):\n        raise ValueError(\'Đường dẫn vượt ngoài thư mục dữ liệu.\')\n    return path\n\n\ndef audio_info(path):\n    with wave.open(str(path), \'rb\') as f:\n        if f.getnchannels() != 1 or f.getsampwidth() != 2 or f.getframerate() != 22050:\n            raise ValueError(\'Clip phải là WAV PCM16 mono 22050 Hz.\')\n        duration = f.getnframes() / f.getframerate()\n        samples = array(\'h\', f.readframes(f.getnframes()))\n    if sys.byteorder != \'little\':\n        samples.byteswap()\n    if not samples:\n        raise ValueError(\'Clip rỗng.\')\n    rms = math.sqrt(sum(x*x for x in samples) / len(samples)) / 32768\n    clipped = sum(abs(x) >= 32760 for x in samples) / len(samples)\n    return dict(duration=duration, rms=rms, clippedFraction=clipped)\n\n\ndef prepare(source, root, model=\'small\', device=\'cpu\'):\n    """ASR runs where this script runs. No upload/API and no automatic approval."""\n    root = Path(root).resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    source = Path(source).resolve()\n    source_sha = digest(source)\n    manifest_path = root / \'review.json\'\n    if manifest_path.exists():\n        previous = json.loads(manifest_path.read_text())\n        if previous[\'sourceSha256\'] != source_sha:\n            raise ValueError(\'Thư mục đã chứa một bản thu khác. Chọn thư mục mới.\')\n        if previous.get(\'status\') == \'awaiting_review\':\n            print(\'Đã tách xong; giữ nguyên các sửa lời và lựa chọn duyệt.\')\n            return\n    originals = root / \'originals\'\n    originals.mkdir(exist_ok=True)\n    original = originals / (\'recording\' + source.suffix.lower())\n    if not original.exists():\n        partial_original = original.with_name(original.name + \'.partial\')\n        shutil.copy2(source, partial_original)\n        if digest(partial_original) != source_sha:\n            raise ValueError(\'Bản sao file gốc chưa hoàn tất.\')\n        partial_original.replace(original)\n    if digest(original) != source_sha:\n        raise ValueError(\'Bản gốc không khớp checksum.\')\n    normalized = root / \'recording-22050.wav\'\n    if not normalized.exists():\n        partial = root / \'recording-22050.partial.wav\'\n        subprocess.run([\'ffmpeg\', \'-nostdin\', \'-v\', \'error\', \'-y\', \'-i\', str(original),\n                        \'-vn\', \'-ac\', \'1\', \'-ar\', \'22050\', \'-c:a\', \'pcm_s16le\', str(partial)], check=True)\n        partial.replace(normalized)\n    from faster_whisper import WhisperModel\n    engine = WhisperModel(model, device=device, compute_type=\'int8\' if device == \'cpu\' else \'float16\')\n    segments, _ = engine.transcribe(str(normalized), language=\'vi\', beam_size=5,\n                                    vad_filter=True, word_timestamps=True,\n                                    condition_on_previous_text=False)\n    # Commit after every completed clip. Resume deterministically without erasing edits.\n    record = json.loads(manifest_path.read_text()) if manifest_path.exists() else dict(\n        schema=1, sourceSha256=source_sha, originalFile=str(original.relative_to(root)),\n        status=\'preparing\', profile=PROFILE, segments=[])\n    known = {s[\'id\']: s for s in record[\'segments\']}\n    clips = root / \'clips\'\n    clips.mkdir(exist_ok=True)\n    def flush(words):\n        if not words:\n            return\n        start = max(0, words[0].start - .08)\n        end = words[-1].end + .12\n        text = \'\'.join(w.word for w in words)\n        sid = f\'{round(start*1000):09d}-{round(end*1000):09d}\'\n        if sid in known:\n            return\n        path = clips / (sid + \'.wav\')\n        if not path.exists():\n            partial = path.with_suffix(\'.partial.wav\')\n            subprocess.run([\'ffmpeg\', \'-nostdin\', \'-v\', \'error\', \'-y\', \'-ss\', str(start),\n                            \'-i\', str(normalized), \'-t\', str(end-start), \'-ac\', \'1\',\n                            \'-ar\', \'22050\', \'-c:a\', \'pcm_s16le\', str(partial)], check=True)\n            partial.replace(path)\n        info = audio_info(path)\n        flags = []\n        if not 2 <= info[\'duration\'] <= 20:\n            flags.append(\'duration\')\n        if info[\'rms\'] < .003:\n            flags.append(\'quiet\')\n        if info[\'clippedFraction\'] > .01:\n            flags.append(\'clipping\')\n        if any(getattr(w, \'probability\', 1) < .5 for w in words):\n            flags.append(\'uncertain_transcript\')\n        if not text.rstrip().endswith((\'.\', \'?\', \'!\', \'…\', \',\', \';\', \':\')):\n            flags.append(\'check_phrase_boundary\')\n        item = dict(id=sid, audio=str(path.relative_to(root)), start=start, end=end,\n                    originalText=text, text=text, approved=False, flags=flags, **info)\n        record[\'segments\'].append(item)\n        known[sid] = item\n        atomic_json(manifest_path, record)\n        if len(record[\'segments\']) % 25 == 0:\n            print(f"PREPARE clips={len(record[\'segments\'])} position={end:.1f}s", flush=True)\n\n    for words in word_groups(segments):\n        flush(words)\n    record[\'status\'] = \'awaiting_review\'\n    atomic_json(manifest_path, record)\n    print(f"Đã tách {len(record[\'segments\'])} clip. Nghe, sửa lời và duyệt trước khi train.")\n\n\ndef make_dataset(root, minimum=30):\n    root = Path(root).resolve()\n    record = json.loads((root / \'review.json\').read_text())\n    if record.get(\'status\') != \'awaiting_review\':\n        raise ValueError(\'Tách audio chưa hoàn tất.\')\n    items = [s for s in record[\'segments\'] if s.get(\'approved\') is True]\n    if len(items) < minimum:\n        raise ValueError(f\'Cần tối thiểu {minimum} clip đã nghe và duyệt.\')\n    seen = set()\n    unique = []\n    for s in sorted(items, key=lambda x: x[\'start\']):\n        if not s[\'text\'].strip() or any(c in s[\'text\'] for c in \'\\r\\n\\0\'):\n            raise ValueError(\'Lời đọc trống hoặc có ký tự xuống dòng/NUL.\')\n        info = audio_info(safe_path(root, s[\'audio\']))\n        if not 2 <= info[\'duration\'] <= 20 or info[\'rms\'] < .003 or info[\'clippedFraction\'] > .01:\n            raise ValueError(f"Clip {s[\'id\']} chưa đạt kiểm tra audio.")\n        s = dict(s, **info)\n        key = s[\'text\'].strip().casefold()\n        if key in seen:\n            continue\n        seen.add(key)\n        unique.append(s)\n    if len(unique) < minimum:\n        raise ValueError(\'Không đủ câu khác nhau sau loại trùng.\')\n    # Last 10% in time forms an untouched listening holdout. Never train on it.\n    count = max(3, math.ceil(len(unique) * .1))\n    train, holdout = unique[:-count], unique[-count:]\n    dataset = root / \'dataset\'\n    (dataset / \'audio\').mkdir(parents=True, exist_ok=True)\n    rows = []\n    for s in train:\n        filename = s[\'id\'] + \'.wav\'\n        shutil.copy2(safe_path(root, s[\'audio\']), dataset / \'audio\' / filename)\n        rows.append((filename, s[\'text\']))\n    temp = dataset / \'metadata.csv.partial\'\n    with temp.open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.writer(f, delimiter=\'|\')\n        writer.writerows(rows)\n    temp.replace(dataset / \'metadata.csv\')\n    atomic_json(dataset / \'holdout.json\', holdout)\n    report = dict(trainClips=len(train), holdoutClips=len(holdout),\n                  trainMinutes=sum(s[\'duration\'] for s in train)/60,\n                  sourceSha256=record[\'sourceSha256\'], reviewedManifestSha256=digest(root/\'review.json\'),\n                  metadataSha256=digest(dataset/\'metadata.csv\'), profile=PROFILE,\n                  audioSha256={name:digest(dataset/\'audio\'/name) for name,_ in rows})\n    atomic_json(dataset / \'report.json\', report)\n    print(json.dumps({k:v for k,v in report.items() if k != \'audioSha256\'}, ensure_ascii=False, indent=2))\n    return report\n\n\ndef review_ui(root):\n    import ipywidgets as w\n    from IPython.display import Audio, display, clear_output\n    root = Path(root)\n    path = root/\'review.json\'\n    data = json.loads(path.read_text())\n    if data.get(\'status\') != \'awaiting_review\' or not data[\'segments\']:\n        raise ValueError(\'Chưa có clip sẵn sàng để duyệt.\')\n    idx = w.BoundedIntText(value=0, min=0, max=len(data[\'segments\'])-1, description=\'Đoạn:\')\n    text = w.Textarea(layout=w.Layout(width=\'95%\', height=\'100px\'))\n    approved = w.Checkbox(description=\'Đã nghe, lời khớp và chỉ một người nói\')\n    save = w.Button(description=\'Lưu và tiếp tục\', button_style=\'success\')\n    out = w.Output()\n    def show(*_):\n        s = data[\'segments\'][idx.value]\n        text.value = s[\'text\']\n        approved.value = s[\'approved\']\n        with out:\n            clear_output(wait=True)\n            print(f"{idx.value+1}/{len(data[\'segments\'])} | {s[\'duration\']:.1f}s | Cờ: {s[\'flags\']}")\n            print(\'Bản chép gốc:\', s[\'originalText\'])\n            display(Audio(filename=str(safe_path(root, s[\'audio\']))))\n    def commit(_):\n        s = data[\'segments\'][idx.value]\n        s[\'text\'], s[\'approved\'] = text.value, approved.value\n        atomic_json(path, data)\n        if idx.value < idx.max:\n            idx.value += 1\n    idx.observe(show, names=\'value\')\n    save.on_click(commit)\n    display(idx, out, text, approved, save)\n    show()\n\n\ndef train(root, base_checkpoint, output, max_steps=300, resume=None):\n    """Fresh fine-tune uses warmstart, never resumes epoch 4769 from base."""\n    import torch\n    from lightning.pytorch import Trainer, seed_everything\n    from lightning.pytorch.callbacks import ModelCheckpoint\n    from piper.train.vits.dataset import VitsDataModule\n    from piper.train.vits.lightning import VitsModel\n    if not torch.cuda.is_available():\n        raise ValueError(\'Chưa có GPU CUDA; không bắt đầu train.\')\n    root, output = Path(root), Path(output)\n    if digest(base_checkpoint) != \'c3879268594a85b5133146e6a4709a5cc2f3014ef0eabf9280f67289054ba2bc\':\n        raise ValueError(\'Checkpoint nền chưa đúng checksum đã khóa.\')\n    report = json.loads((root/\'dataset/report.json\').read_text())\n    if report[\'metadataSha256\'] != digest(root/\'dataset/metadata.csv\'):\n        raise ValueError(\'Dataset đã thay đổi. Tạo lại dataset trước khi train.\')\n    if report[\'reviewedManifestSha256\'] != digest(root/\'review.json\'):\n        raise ValueError(\'Lời hoặc trạng thái duyệt đã đổi. Chốt lại dataset trước khi train.\')\n    if report.get(\'profile\') != PROFILE:\n        raise ValueError(\'Dataset chưa được chốt theo cấu hình giữ giọng gốc.\')\n    for name, sha in report.get(\'audioSha256\', {}).items():\n        if digest(safe_path(root/\'dataset/audio\', name)) != sha:\n            raise ValueError(\'Audio đã thay đổi sau khi duyệt. Chốt lại dataset.\')\n    output.mkdir(parents=True, exist_ok=True)\n    seed_everything(42, workers=True)\n    # Official checkpoint only; hash is verified by the notebook before this call.\n    checkpoint = torch.load(base_checkpoint, map_location=\'cpu\', weights_only=False)\n    hp = checkpoint[\'hyper_parameters\']\n    architecture = [\'resblock\',\'resblock_kernel_sizes\',\'resblock_dilation_sizes\',\n                    \'upsample_rates\',\'upsample_initial_channel\',\'upsample_kernel_sizes\',\n                    \'filter_length\',\'hop_length\',\'win_length\',\'mel_channels\',\'mel_fmin\',\'mel_fmax\',\n                    \'inter_channels\',\'hidden_channels\',\'filter_channels\',\'n_heads\',\'n_layers\',\n                    \'kernel_size\',\'p_dropout\',\'n_layers_q\',\'use_spectral_norm\',\'gin_channels\',\n                    \'use_sdp\',\'segment_size\',\'num_symbols\']\n    options = {k: hp[k] for k in architecture if k in hp}\n    model = VitsModel(**options, batch_size=4, sample_rate=22050, num_speakers=1,\n                      learning_rate=PROFILE[\'learningRate\'], learning_rate_d=PROFILE[\'learningRate\'], mos_metric=None)\n    # Strict load catches architectural mismatch instead of silently training random layers.\n    model.load_state_dict(checkpoint[\'state_dict\'], strict=True)\n    del checkpoint\n    dm = VitsDataModule(csv_path=root/\'dataset/metadata.csv\', audio_dir=root/\'dataset/audio\',\n                       voice_name=\'windi_vi_narrator\', espeak_voice=\'vi\',\n                       cache_dir=root/(\'piper-cache-\'+PROFILE[\'id\']), config_path=output/\'voice.onnx.json\',\n                       sample_rate=22050, batch_size=4, num_workers=2,\n                       validation_split=.1, num_test_examples=0, trim_silence=PROFILE[\'trimSilence\'],\n                       **{k:options[k] for k in [\'num_symbols\',\'filter_length\',\'hop_length\',\'win_length\',\'segment_size\'] if k in options})\n    latest = ModelCheckpoint(dirpath=output/\'checkpoints\', filename=\'step-{step}\',\n                             every_n_train_steps=100, save_last=True, save_top_k=1,\n                             enable_version_counter=False)\n    best = ModelCheckpoint(dirpath=output/\'best\', monitor=\'val_mel\', mode=\'min\', save_top_k=2,\n                           filename=\'epoch-{epoch}-mel-{val_mel:.3f}\', auto_insert_metric_name=False)\n    atomic_json(output/\'run.json\', dict(dataset=report,maxSteps=max_steps,seed=42,\n                                       runtime=\'piper-tts==1.8.0\',status=\'running\',profile=PROFILE))\n    trainer = Trainer(accelerator=\'gpu\', devices=1, precision=\'32-true\', max_steps=max_steps,\n                      max_epochs=-1, default_root_dir=output, callbacks=[latest,best],\n                      log_every_n_steps=10, num_sanity_val_steps=1, deterministic=False)\n    try:\n        trainer.fit(model, datamodule=dm, ckpt_path=resume)\n    finally:\n        # Lightning writes normal periodic checkpoints; retain an explicit interruption point too.\n        if trainer.global_step > 0:\n            partial = output/\'interrupted.partial.ckpt\'\n            trainer.save_checkpoint(partial)\n            partial.replace(output/\'interrupted.ckpt\')\n\n\ndef listening_pack(root, model, output, count=5):\n    """Private A/B references only. Never include these recordings in public exports."""\n    from piper import PiperVoice\n    root, model, output = Path(root), Path(model), Path(output)\n    holdout = json.loads((root/\'dataset/holdout.json\').read_text())\n    if len(holdout) < 1:\n        raise ValueError(\'Chưa có phần nghe đối chiếu giữ riêng.\')\n    # Reject changed/non-matching references instead of comparing against stale text/audio.\n    report = json.loads((root/\'dataset/report.json\').read_text())\n    if report[\'reviewedManifestSha256\'] != digest(root/\'review.json\'):\n        raise ValueError(\'Phần đối chiếu đã thay đổi; chốt lại dataset.\')\n    output.mkdir(parents=True, exist_ok=True)\n    total = min(count, len(holdout))\n    indices = sorted({round(i*(len(holdout)-1)/max(1,total-1)) for i in range(total)})\n    voice = PiperVoice.load(model)\n    results = []\n    for index in indices:\n        s = holdout[index]\n        reference = output/(s[\'id\']+\'-original.wav\')\n        generated = output/(s[\'id\']+\'-generated.wav\')\n        temp = generated.with_suffix(\'.partial.wav\')\n        with wave.open(str(temp),\'wb\') as wav:\n            voice.synthesize_wav(s[\'text\'],wav)\n        info = audio_info(temp)\n        if info[\'rms\'] < .003:\n            raise ValueError(\'Bản tạo ra chưa có tiếng hợp lệ.\')\n        temp.replace(generated)\n        shutil.copy2(safe_path(root,s[\'audio\']),reference)\n        results.append(dict(text=s[\'text\'],original=reference.name,generated=generated.name,\n                            originalDuration=s[\'duration\'],generatedDuration=info[\'duration\'],\n                            sourceSha256=report[\'sourceSha256\'],modelSha256=digest(model),\n                            sameVoice=None,pronunciationCorrect=None,rhythmNatural=None,\n                            notes=\'\',status=\'needs_listening\'))\n        atomic_json(output/\'comparison.json\',results)\n    print(\'Đã tạo bộ nghe riêng: bản gốc và bản tạo cùng lời. Chưa chấm độ giống giọng.\')\n    return results\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    sub = parser.add_subparsers(dest=\'action\', required=True)\n    prep = sub.add_parser(\'prepare\')\n    prep.add_argument(\'source\'); prep.add_argument(\'--root\',required=True)\n    prep.add_argument(\'--model\',default=\'small\'); prep.add_argument(\'--device\',default=\'cpu\')\n    ds = sub.add_parser(\'dataset\'); ds.add_argument(\'--root\',required=True)\n    args = parser.parse_args()\n    if args.action == \'prepare\':\n        prepare(args.source,args.root,args.model,args.device)\n    else:\n        make_dataset(args.root)\n\n\nif __name__ == \'__main__\':\n    main()\n'
(ROOT/"toolkit.py").write_text(TOOLKIT, encoding="utf-8")
print("Đã ghi công cụ CapCut sạch, giữ giọng gốc. Không có bản thu trong notebook.")


## 3. Tải checkpoint cố định, xác minh và kiểm tra tương thích
Checkpoint train khoảng 846 MB; **không phải dung lượng model ONNX cuối cùng**. Chỉ tải trọng số, cấu hình và attribution, không tải dữ liệu thu âm gốc của bộ VAIS.


In [ ]:
BASE = ROOT/'base'
BASE.mkdir(exist_ok=True)
REVISION = '37d1bea250e21ca0d617818d2bef1db724edb80c'
PREFIX = f'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/{REVISION}/vi/vi_VN/vais1000/medium/'
BASE_CKPT = BASE/'vais1000-medium.ckpt'
def fetch_verified(url,path,expected_sha=None):
    def sha(p):
        h=hashlib.sha256()
        with p.open('rb') as f:
            for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
        return h.hexdigest()
    if not path.exists() or (expected_sha and sha(path)!=expected_sha):
        tmp=path.with_name(path.name+'.partial')
        with urllib.request.urlopen(url,timeout=120) as incoming, tmp.open('wb') as out:
            while True:
                b=incoming.read(1024*1024)
                if not b: break
                out.write(b)
        if expected_sha and sha(tmp)!=expected_sha: raise RuntimeError('Checksum không khớp; không dùng checkpoint.')
        tmp.replace(path)
    return sha(path)
base_sha=fetch_verified(PREFIX+'epoch%3D4769-step%3D919580.ckpt', BASE_CKPT,
    'c3879268594a85b5133146e6a4709a5cc2f3014ef0eabf9280f67289054ba2bc')
fetch_verified(PREFIX+'config.json',BASE/'donor-config.json')
fetch_verified(PREFIX+'MODEL_CARD',BASE/'MODEL_CARD')
check = r"""
import torch, importlib.metadata, json
from pathlib import Path
from piper.train.vits.lightning import VitsModel
from piper.train.vits.dataset import VitsDataModule
from piper.train.vits.monotonic_align import maximum_path
p=Path('/content/windi-small-voice')
ck=torch.load(p/'base/vais1000-medium.ckpt',map_location='cpu',weights_only=False)
hp=ck['hyper_parameters']
architecture=['resblock','resblock_kernel_sizes','resblock_dilation_sizes','upsample_rates',
'upsample_initial_channel','upsample_kernel_sizes','filter_length','hop_length','win_length',
'mel_channels','mel_fmin','mel_fmax','inter_channels','hidden_channels','filter_channels',
'n_heads','n_layers','kernel_size','p_dropout','n_layers_q','use_spectral_norm','gin_channels',
'use_sdp','segment_size','num_symbols']
m=VitsModel(**{k:hp[k] for k in architecture if k in hp},sample_rate=22050,num_speakers=1,mos_metric=None)
m.load_state_dict(ck['state_dict'],strict=True)
report=dict(piper=importlib.metadata.version('piper-tts'),torch=torch.__version__,
    cudaAvailable=torch.cuda.is_available(),checkpointStrictLoad=True,
    checkpointEpoch=ck.get('epoch'),sampleRate=22050,modelBytes=(p/'base/vais1000-medium.ckpt').stat().st_size)
(p/'setup-proof.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
"""
subprocess.run([str(PY),'-c',check],check=True)
with (ROOT/'environment-lock.txt').open('w') as f:
    subprocess.run([str(PY),'-m','pip','freeze'],check=True,stdout=f)
print('Checkpoint khớp kiến trúc. Bắt đầu fine-tune mới, không tiếp tục epoch cũ của tác giả.')


## 4. Chuẩn bị model chép lời local trong Colab
Model ASR chỉ phục vụ chuẩn bị dữ liệu; không đưa vào app đọc sách. Bước này tải model từ Hugging Face lần đầu, không gửi audio tới API.


In [ ]:
subprocess.run([str(PY),'-c',
    "from faster_whisper import WhisperModel; m=WhisperModel('small',device='cpu',compute_type='int8'); print('ASR small đã sẵn sàng.')"],check=True)
print((ROOT/'setup-proof.json').read_text())
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
compile((ROOT/'toolkit.py').read_text(),str(ROOT/'toolkit.py'),'exec')
print('Công cụ hợp lệ. ASR đã tải. Chưa nhận file và chưa train.')


## 5. Nơi lưu bền vững — thực hiện khi có dữ liệu
Colab có thể xóa máy ảo. Trước khi train, cần lưu bản thu, lời duyệt và checkpoint vào thư mục riêng trên Drive. Ô này **mặc định tắt**; khi bật, Google yêu cầu bạn tự cho phép mount Drive. Quyền mount có thể cho mã notebook truy cập Drive, không chỉ thư mục huấn luyện. Không cần thực hiện ở giai đoạn chờ file.


In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/WindiSachNoi-Training/voice-001')
print('Thư mục dự kiến:', WORK)
print('Drive sẵn sàng:', Path('/content/drive/MyDrive').is_dir())


## 6. Nhận file bạn chọn
Bật khi đã có bản thu. Chỉ chọn một file WAV/M4A/MP3/FLAC. Audio sẽ được xử lý trên máy Colab của tài khoản này; không chia sẻ công khai. Giữ file gốc trên Mac nữa.


In [ ]:
UPLOAD_RECORDING = False
if UPLOAD_RECORDING:
    if not Path('/content/drive/MyDrive').is_dir(): raise RuntimeError('Chọn nơi lưu bền vững ở bước 5 trước.')
    from google.colab import files
    incoming=files.upload()
    if len(incoming)!=1: raise RuntimeError('Chỉ chọn một bản thu.')
    import shutil
    name=next(iter(incoming))
    if Path(name).suffix.lower() not in {'.wav','.m4a','.mp3','.flac','.aac','.ogg'}: raise RuntimeError('Định dạng chưa hỗ trợ.')
    WORK.mkdir(parents=True,exist_ok=True)
    SOURCE=WORK/Path(name).name
    if SOURCE.exists(): raise RuntimeError('File đã tồn tại; không ghi đè bản gốc.')
    shutil.copy2(Path(name),SOURCE)
    print('Đã lưu bản thu được chọn:',SOURCE)
else:
    print('Đang chờ file của bạn. Không có audio nào được tải lên.')


## 7. Tách và chép lời
Xử lý trên máy Colab, lưu kết quả từng clip về Drive. Nếu ngắt, chạy lại với cùng file để tiếp tục; giữ các đoạn đã duyệt. Không tự khử nhạc hay tạo lại giọng gốc.


In [ ]:
PREPARE_AUDIO = False
if PREPARE_AUDIO:
    if 'SOURCE' not in globals() or not SOURCE.is_file(): raise RuntimeError('Chưa chọn bản thu.')
    subprocess.run([str(PY),str(ROOT/'toolkit.py'),'prepare',str(SOURCE),'--root',str(WORK)],check=True)
else:
    print('Chưa tách audio: chờ bản thu được chọn.')


## 8. Nghe và duyệt lời từng clip
Giữ `originalText` riêng; sửa `text` cho đúng lời, đủ dấu tiếng Việt. Chỉ duyệt clip một người nói, không nhạc, không tiếng người khác. Không có nút tự duyệt toàn bộ.


In [ ]:
import importlib.util
spec=importlib.util.spec_from_file_location('windi_training',ROOT/'toolkit.py')
kit=importlib.util.module_from_spec(spec); spec.loader.exec_module(kit)
if (WORK/'review.json').exists(): kit.review_ui(WORK)
else: print('Chưa có bản thu để duyệt.')


## 9. Chốt dataset
Chỉ lấy clip đã duyệt. Loại câu trùng, giữ 10% cuối bản thu làm bài nghe riêng, không đưa vào train. Piper giữ thêm 10% nội bộ cho validation. Chưa đủ 30 clip duyệt thì không tạo dataset.


In [ ]:
BUILD_DATASET = False
if BUILD_DATASET: kit.make_dataset(WORK)
else: print('Chưa chốt dataset: chờ nghe và duyệt.')


## 10. Train thử có giới hạn
**Chỉ bật khi dữ liệu đã duyệt và bạn muốn bắt đầu dùng GPU.** Mặc định thử 300 bước, learning rate 3e-5, batch 4, FP32 để tương thích T4, tắt cắt khoảng lặng. Đây là pilot kiểm tra hướng học giọng, không phải số bước train đủ cho mọi bản thu. Lưu mỗi 100 bước và khi dừng có kiểm soát; ngắt đột ngột chỉ giữ checkpoint đã ghi xong. Chọn lại `RESUME` để tiếp tục. Đánh giá bằng nghe, không tự coi loss thấp là giọng đạt chất lượng.


In [ ]:
START_TRAINING = False
MAX_STEPS = 300
RESUME = None  # ví dụ: str(WORK/'run/checkpoints/last.ckpt')
if START_TRAINING:
    if not Path('/content/drive/MyDrive').is_dir(): raise RuntimeError('Chưa có nơi lưu checkpoint bền vững.')
    if not (WORK/'dataset/report.json').is_file(): raise RuntimeError('Chưa có dataset đã duyệt.')
    command = ("import sys; sys.path.insert(0,"+repr(str(ROOT))+"); from toolkit import train; train("+
        repr(str(WORK))+","+repr(str(BASE_CKPT))+","+repr(str(WORK/'run'))+
        ",max_steps="+repr(MAX_STEPS)+",resume="+repr(RESUME)+")")
    subprocess.run([str(PY),'-c',command],check=True)
else:
    print('Train đang TẮT. Không dùng GPU để train khi chưa có dữ liệu.')


## 11. Xuất ONNX và đóng gói để thử trên Mac
Chọn checkpoint sau khi nghe; không tự chọn bản có loss thấp nhất. Gói gồm model, cấu hình, nguồn/giấy phép và bài nghe đối chiếu. Không có file thu gốc. Export chưa đồng nghĩa với đã kiểm thử chạy trong app.


In [ ]:
EXPORT_MODEL = False
SELECTED_CKPT = WORK/'run/checkpoints/last.ckpt'
if EXPORT_MODEL:
    if not SELECTED_CKPT.is_file(): raise RuntimeError('Chưa có checkpoint để xuất.')
    OUT=WORK/'export'; OUT.mkdir(exist_ok=True)
    temp=OUT/'voice.partial.onnx'
    subprocess.run([str(PY),'-m','piper.train.export_onnx','--checkpoint',str(SELECTED_CKPT),
        '--output-file',str(temp)],check=True)
    subprocess.run([str(PY),'-c',"import onnx,sys; onnx.checker.check_model(sys.argv[1])",str(temp)],check=True)
    temp.replace(OUT/'voice.onnx')
    shutil.copy2(WORK/'run/voice.onnx.json',OUT/'voice.onnx.json')
    shutil.copy2(BASE/'MODEL_CARD',OUT/'DONOR_MODEL_CARD.txt')
    shutil.copy2(ROOT/'environment-lock.txt',OUT/'environment-lock.txt')
    (OUT/'ATTRIBUTION.txt').write_text('Piper: https://github.com/OHF-Voice/piper1-gpl (GPL-3.0)\nDonor VAIS1000: '+PREFIX+'MODEL_CARD\nDataset license in donor model card: CC BY 4.0\nPrivate user recording not included.\n')
    (OUT/'export-info.json').write_text(json.dumps(dict(sourceCheckpoint=str(SELECTED_CKPT.name),
        sha256=kit.digest(OUT/'voice.onnx'),bytes=(OUT/'voice.onnx').stat().st_size,
        checkpointRevision=REVISION,status='exported-not-yet-validated-on-mac'),indent=2))
    package=shutil.make_archive(str(WORK/'windi-voice-001'),'zip',OUT)
    print('Gói thử local:',package)
else:
    print('Chưa có model đã train để xuất.')


## 12. Nghe đối chiếu giọng gốc — chỉ sau khi có model
Tạo 5 cặp audio cùng lời từ phần holdout chưa đưa vào train. Nghe độ giống âm sắc, phát âm, giọng vùng miền và nhịp đọc. Bộ này có audio tham chiếu riêng tư, không nằm trong gói ONNX xuất ở trên. Không tự tăng/giảm tốc hay nâng cao độ để làm kết quả có vẻ giống. Chưa có file thật nên chưa đo độ giống giọng.


In [ ]:
MAKE_COMPARISON = False
if MAKE_COMPARISON:
    kit.listening_pack(WORK, WORK/'export/voice.onnx', WORK/'private-comparison')
else:
    print('Chờ giọng đã train để tạo bài nghe A/B với bản gốc.')
